In [234]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import json
import re
import time

options = Options()
# options.page_load_strategy = 'eager'
driver = webdriver.Chrome()

In [235]:
def save_data(path, data):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

In [236]:
def get_comments(driver):
    cmt = {}
    count=0
    cmt_section = WebDriverWait(driver, 20).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, 'ol#reviewSectionComments'))
    )
    
    cmt_els = cmt_section.find_elements(By.TAG_NAME, 'li')
    
    for cmt_el in cmt_els:
      
        cmt_details = cmt_el.find_element(By.CSS_SELECTOR, '[data-element-name="review-comment"]').text.split('\n')
        
        reviewer_name = cmt_details[2]
        group = cmt_details[3]
        room_type = cmt_details[4]
        stay = cmt_details[5]
        
        cmt_content = cmt_el.find_element(By.CLASS_NAME, 'Review-comment-body').text.strip()
        cmt_at = cmt_el.find_element(By.CSS_SELECTOR, 'div.Review-comment-body').find_element(By.XPATH, './following-sibling::div[1]').find_element(By.TAG_NAME, 'span').text

        
        cmt[reviewer_name] ={
            'diem_danh_gia': cmt_details[0] + ' '+ cmt_details[1],
            'noi_dung_comment': cmt_content,
            'loai_hinh_du_lich':group,
            'loai_phong': room_type,
            'luu_tru': stay,
            'thoi_diem_comment':cmt_at
        }
        
        count+=1
    
    return cmt, count

In [237]:
url = 'https://www.agoda.com/vi-vn/wyndham-hoi-an-royal-beachfront-resort-h69359304/hotel/hoi-an-vn.html'
driver.get(url)
ActionChains(driver).send_keys(Keys.ESCAPE).perform()

data = {}

name = driver.find_element(By.CSS_SELECTOR, '[data-selenium="hotel-header-name"]').text
data['ten_khach_san'] = name

star = driver.find_element(By.CSS_SELECTOR, '[data-selenium="mosaic-hotel-rating"]').text
data['sao']  = star

rating = driver.find_element(By.CSS_SELECTOR, '[data-testid="review-plate-redesign-score"]').text
data['diem_danh_gia_tong_the'] = rating.split('\n')[0]


In [238]:
# get total number of reviews for stopping condition
num_comments_text = driver.find_element(By.CSS_SELECTOR, '[data-testid="review-plate-redesign-score"]').text
num_comments = max([int(i) for i in re.findall(r'(\d+)', num_comments_text)])

print(f'Tổng số comments hiển thị: {num_comments}')

cmt_scraped=0

while cmt_scraped < num_comments:
    cmt, count = get_comments(driver)
    if cmt is not None:
        data.update(cmt)
        
        cmt_scraped+=count
        print(f'Số lượng comment đã cào: {cmt_scraped}')
    
    next_btn = driver.find_element(By.CSS_SELECTOR, '[data-element-name="review-paginator-next"]')
    
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_btn)
    
    if next_btn.is_enabled() is False:
        print("Không thể chuyển tiếp trang, dừng lại")
        break
    
    next_btn.click()
    print('Đợi để trang mới load')
    time.sleep(5)
    
        
save_data('data_bo_sung.json', data)
    

Tổng số comments hiển thị: 127
Số lượng comment đã cào: 5
Đợi để trang mới load
Số lượng comment đã cào: 10
Đợi để trang mới load
Số lượng comment đã cào: 15
Đợi để trang mới load
Số lượng comment đã cào: 20
Đợi để trang mới load
Số lượng comment đã cào: 25
Đợi để trang mới load
Số lượng comment đã cào: 30
Không thể chuyển tiếp trang, dừng lại
